In [ ]:
!pip install -q transformers accelerate torch torchvision
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q groq
!pip install -q gradio

import torch
from PIL import Image
import gradio as gr
import faiss
import numpy as np
from transformers import BlipProcessor, BlipForConditionalGeneration
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

dimension = 384  # MiniLM embedding dimension
index = faiss.IndexFlatL2(dimension)
stored_chunks = []

GROQ_API_KEY = "YOUR_GROQ_API_KEY"

client = Groq(api_key=GROQ_API_KEY)

def generate_caption(image):
    image = Image.fromarray(image).convert("RGB")
    inputs = processor(image, return_tensors="pt").to(device)
    output = blip_model.generate(**inputs)
    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

def store_in_vector_db(text):
    global stored_chunks
    
    chunks = [text[i:i+200] for i in range(0, len(text), 200)]
    embeddings = embedding_model.encode(chunks)
    
    index.add(np.array(embeddings))
    stored_chunks.extend(chunks)
    
def retrieve_context(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)
    
    retrieved = [stored_chunks[i] for i in indices[0] if i < len(stored_chunks)]
    return " ".join(retrieved)

def ask_llm(context, question):
    prompt = f"""
    You are a Vision RAG assistant.
    Use the following image context to answer.

    Context:
    {context}

    Question:
    {question}

    Answer clearly and precisely.
    """

    completion = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    return completion.choices[0].message.content

def vision_rag_pipeline(image, question):
    global index, stored_chunks
    
    if image is None:
        return "Please upload an image."
    
    # Reset vector store for new image
    index.reset()
    stored_chunks = []
    
    caption = generate_caption(image)
    store_in_vector_db(caption)
    
    context = retrieve_context(question)
    answer = ask_llm(context, question)
    
    return f"📷 Caption: {caption}\n\n🤖 Answer: {answer}"

custom_css = """
body {
    background-color: #0f172a;
}
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    
    gr.Markdown(
        """
        # 🔍 Vision RAG Assistant  
        Upload an image and ask questions about it.
        Powered by BLIP + Llama 3 (Groq) + FAISS
        """
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="numpy", label="Upload Image")
            question_input = gr.Textbox(
                placeholder="Ask something about the image...",
                label="Your Question"
            )
            submit_btn = gr.Button("Analyze", variant="primary")
        
        with gr.Column(scale=1):
            output = gr.Textbox(label="Vision RAG Response", lines=15)
    
    submit_btn.click(
        vision_rag_pipeline,
        inputs=[image_input, question_input],
        outputs=output
    )

demo.launch(debug=True)
